# E6: ALFWorld with GPT-4o-mini — Cost-Effective Scaling Test

Same protocol as E5 but with GPT-4o-mini (~15x cheaper than GPT-4o).

**Research question**: Can CVX memory compensate for a weaker model?
If CVX + 4o-mini approaches NoMemory + 4o, memory acts as a model-size substitute.

| Condition | Model | Cost/1M tokens |
|-----------|-------|---------------|
| E5 NoMemory | GPT-4o | ~$2.50 |
| E5 CVX-Causal | GPT-4o | ~$2.50 |
| **E6 NoMemory** | **GPT-4o-mini** | **~$0.15** |
| **E6 CVX-Causal** | **GPT-4o-mini** | **~$0.15** |


In [ ]:
import os, json, time, re
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from alfworld.agents.environment.alfred_tw_env import AlfredTWEnv

load_dotenv()

# === CONFIG ===
USE_OLLAMA = False  # Set True for local qwen2.5 via Ollama
OLLAMA_MODEL = "qwen2.5-coder:7b-instruct"
OLLAMA_URL = "http://hpc.glaciar.lab:11434/v1"

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key="ollama")
    LLM_MODEL = OLLAMA_MODEL
else:
    api_key = os.environ.get("OPENAI_API_KEY") or os.environ.get("OPENAI_TOKEN")
    if not api_key:
        raise ValueError("Set OPENAI_API_KEY or OPENAI_TOKEN in .env")
    client = OpenAI(api_key=api_key)
    LLM_MODEL = "gpt-4o-mini"  # Change to "gpt-4o-mini" for cost-effective run

EMBED_MODEL = "all-MiniLM-L6-v2"
MAX_STEPS = 30
N_EVAL = 50  # More games than E3 (30) for statistical power
TOP_K = 3

DATA_DIR = Path("../data/episodic")
embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f"Embedding: {EMBED_MODEL} (D={D})")
print(f"LLM: {LLM_MODEL}")
print(f"Max steps: {MAX_STEPS}, Eval games: {N_EVAL}")



## 1. Load CVX Episodic Memory (from E2)

We reuse the index built in E2 from AgentInstruct expert trajectories.

In [ ]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / "alfworld_episodic.cvx")

try:
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f"Loaded CVX index: {len(index)} points")
except Exception as e:
    print(f"Load failed ({e}), rebuilding from E2...")
    # Fallback: rebuild from E2 data if available
    index = None

# Apply centering for better retrieval (RFC-012)
if index is not None:
    centroid = index.compute_centroid()
    if centroid:
        index.set_centroid(centroid)
        print(f"Centroid set (D={len(centroid)})")


## 2. Initialize ALFWorld Environment

In [ ]:
config = {
    'dataset': {
        'data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/train'),
        'eval_id_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_seen'),
        'eval_ood_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_unseen'),
        'num_train_games': 0,
        'num_eval_games': 0,
    },
    'env': {
        'goal_desc_human_anns_prob': 0.0,
        'task_types': [1, 2, 3, 4, 5, 6],
        'domain_randomization': False,
        'expert_type': 'handcoded',
    },
    'general': {'training_method': 'dqn', 'random_seed': 42},
    'rl': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'dagger': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'logic': {
        'domain': os.path.expanduser('~/.cache/alfworld/logic/alfred.pddl'),
        'grammar': os.path.expanduser('~/.cache/alfworld/logic/alfred.twl2'),
    }
}

env_wrapper = AlfredTWEnv(config, train_eval='eval_out_of_distribution')
env = env_wrapper.init_env(batch_size=1)

# Quick test
obs, info = env.reset()
task = obs[0].split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in obs[0] else 'unknown'
print(f'Test game: {task}')
print(f'Available actions: {len(info["admissible_commands"][0])}')
print(f'Total eval games: {len(env_wrapper.game_files)}')

## 3. CVX Causal Retrieval (Step-by-Step)

The key difference from E2: the query is the **current observation**, not the task description.
This finds similar mid-episode states and returns what successful agents did next.

In [ ]:
def retrieve_causal(observation, task, top_k=TOP_K):
    """Search CVX for similar past states using native causal_search."""
    query_text = f"{task} | {observation}"
    query_vec = embedder.encode(query_text).tolist()
    query_vec = index.centered_vector(query_vec)
    
    results = index.causal_search(
        vector=query_vec,
        k=top_k,
        temporal_context=5,
    )
    return results


def format_causal_context(results):
    """Format causal search results for LLM prompt."""
    if not results:
        return ""
    
    parts = ["Past experience from similar situations:"]
    for i, r in enumerate(results[:TOP_K]):
        n_succ = len(r.get("successors", []))
        parts.append(f"  [{i+1}] Similar state (score={r['score']:.2f}), {n_succ} next steps")
    
    return "\n".join(parts) + "\n"


## 4. LLM Agent

The agent receives the observation, admissible actions, and optionally CVX causal context.
It must output exactly one of the admissible actions.

In [ ]:
def agent_act(observation, admissible_actions, task, history, causal_context=''):
    """LLM agent: choose one admissible action."""
    system = (
        'You are an agent in a household environment. Your goal is to complete the task. '
        'You MUST choose exactly ONE action from the list of available actions. '
        'Output ONLY the action text, nothing else.'
    )
    
    user_parts = []
    user_parts.append(f'Task: {task}\n')
    
    if causal_context:
        user_parts.append(causal_context)
    
    if history:
        user_parts.append('Recent actions:')
        for h in history[-5:]:  # last 5 steps
            user_parts.append(f'  > {h["action"]} → {h["obs"][:80]}')
        user_parts.append('')
    
    user_parts.append(f'Current observation: {observation}\n')
    user_parts.append('Available actions:')
    for a in admissible_actions:
        user_parts.append(f'  - {a}')
    user_parts.append('\nChoose ONE action:')
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=0.0,
        max_tokens=100,
    )
    
    chosen = response.choices[0].message.content.strip()
    
    # Fuzzy match: find the closest admissible action
    chosen_lower = chosen.lower().strip()
    for action in admissible_actions:
        if action.lower() == chosen_lower:
            return action
    
    # Partial match
    for action in admissible_actions:
        if action.lower() in chosen_lower or chosen_lower in action.lower():
            return action
    
    # Fallback: first action
    return admissible_actions[0]


def run_episode(env, use_memory=False, verbose=False):
    """Run one episode with or without CVX causal memory."""
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    
    history = []
    
    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        
        # CVX causal retrieval (if enabled)
        causal_context = ''
        if use_memory:
            retrievals = retrieve_causal(observation, task)
            causal_context = format_causal_context(retrievals)
        
        # Agent chooses action
        action = agent_act(observation, admissible, task, history, causal_context)
        
        if verbose:
            print(f'  Step {step+1}: {action}')
        
        # Execute
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        
        history.append({'action': action, 'obs': observation[:100]})
        
        if dones[0]:
            break
    
    won = info['won'][0]
    return {
        'task': task,
        'won': won,
        'steps': len(history),
        'history': history,
    }


# Test one episode with memory
print('=== Test Episode (with CVX-Causal) ===\n')
result = run_episode(env, use_memory=True, verbose=True)
print(f'\nTask: {result["task"]}')
print(f'Won: {result["won"]}, Steps: {result["steps"]}')

## 5. Evaluation

Run N games for each condition and compare task completion rates.

In [ ]:
conditions = {
    "NoMemory": False,
    "CVX-Causal": True,
}

all_results = {}
for cond_name, use_mem in conditions.items():
    print(f"\n{'='*60}")
    print(f"Condition: {cond_name} (model={LLM_MODEL})")
    print(f"{'='*60}")
    
    results = []
    for game_idx in range(N_EVAL):
        try:
            result = run_episode(env, use_memory=use_mem, verbose=False)
            results.append(result)
            status = "win" if result["won"] else "fail"
            print(f"  Game {game_idx+1}/{N_EVAL}: {status} ({result['steps']} steps)")
        except Exception as e:
            print(f"  Game {game_idx+1}/{N_EVAL}: ERROR - {e}")
            results.append({"task": "", "won": False, "steps": MAX_STEPS})
    
    wins = sum(1 for r in results if r["won"])
    rate = wins / len(results) * 100
    print(f"\n  Result: {wins}/{len(results)} = {rate:.1f}%")
    all_results[cond_name] = results


In [ ]:
import plotly.graph_objects as go

colors = {'NoMemory': '#95a5a6', 'CVX-Causal': '#2ecc71'}

# Completion rate bar chart
fig = go.Figure()
for cond, res in all_results.items():
    rate = sum(r['won'] for r in res) / len(res)
    fig.add_trace(go.Bar(
        x=[cond], y=[rate * 100],
        text=f'{rate:.1%}', textposition='outside',
        marker_color=colors.get(cond, '#333'),
        name=cond,
    ))

fig.update_layout(
    title=f'ALFWorld Task Completion Rate (n={N_EVAL}, max_steps={MAX_STEPS})',
    yaxis_title='Completion Rate (%)', yaxis=dict(range=[0, 100]),
    height=400, showlegend=False,
)
fig.show()

# Step distribution
fig2 = go.Figure()
for cond in conditions:
    steps = [r['steps'] for r in all_results[cond]]
    fig2.add_trace(go.Histogram(
        x=steps, name=cond, marker_color=colors.get(cond),
        opacity=0.7, nbinsx=15,
    ))

fig2.update_layout(
    title='Steps per Episode Distribution',
    xaxis_title='Steps', yaxis_title='Count',
    barmode='overlay', height=350,
)
fig2.show()

In [ ]:
from scipy import stats

# McNemar's test
a_won = np.array([r['won'] for r in all_results['CVX-Causal']])
b_won = np.array([r['won'] for r in all_results['NoMemory']])

n_10 = np.sum(a_won & ~b_won)  # CVX-Causal only
n_01 = np.sum(~a_won & b_won)  # NoMemory only
n_11 = np.sum(a_won & b_won)
n_00 = np.sum(~a_won & ~b_won)

if n_10 + n_01 > 0:
    chi2 = (abs(n_10 - n_01) - 1) ** 2 / (n_10 + n_01)
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
else:
    chi2, p_val = 0, 1.0

sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

print('=== McNemar\'s Test: CVX-Causal vs NoMemory ===')
print(f'CVX-Causal only won: {n_10}')
print(f'NoMemory only won:   {n_01}')
print(f'Both won:            {n_11}')
print(f'Neither won:         {n_00}')
print(f'Net: {n_10 - n_01:+d} tasks, chi2={chi2:.2f}, p={p_val:.4f} {sig}')

# Summary
print(f'\n=== E3: INTERACTIVE ALFWORLD RESULTS ===')
print(f'Model: {LLM_MODEL}')
print(f'Memory: {len(episode_data)} expert episodes ({len(index)} vectors)')
print(f'Eval: {N_EVAL} games, max {MAX_STEPS} steps, eval_out_of_distribution')
print(f'\nNoMemory:   {sum(r["won"] for r in all_results["NoMemory"])}/{N_EVAL} = '
      f'{sum(r["won"] for r in all_results["NoMemory"])/N_EVAL:.1%}')
print(f'CVX-Causal: {sum(r["won"] for r in all_results["CVX-Causal"])}/{N_EVAL} = '
      f'{sum(r["won"] for r in all_results["CVX-Causal"])/N_EVAL:.1%}')
print(f'\nCVX features: search (step-by-step), episode encoding, temporal continuation')
print('This is the INTERACTIVE loop — query changes at each step based on env observation.')